In [7]:
import requests
import mysql.connector
from datetime import date
import re
from collections import defaultdict

# ==========================================
# 1. 설정
# ==========================================
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "root",
    "database": "cooking_db",
    "charset": "utf8mb4"
}

API_KEY = "766d4747586e686a36357645785655"
TARGET_MARKET_KEYWORD = "이마트" 

NON_FOOD_ITEMS = [
    '치약', '비누', '사이다', '콜라', '맥주', '소주', '샴푸', '린스', '화장지', 
    '세제', '생수', '컵라면', '건전지', '담배', '부탄가스', '종량제', '생리대', 
    '락스', '바디워시', '칫솔', '로션', '염색약', '섬유유연제', '주방세제',
    '물티슈', '키친타올', '호일', '랩', '위생장갑', '마스크', '소독제', '살충제',
    '가글', '표백제', '탈취제', '건조기', '스타킹', '내복'
]

REMOVE_KEYWORDS = [
    'CJ', '오뚜기', '청정원', '풀무원', '동원', '하림', '농심', '서울우유', '매일', '남양', 
    '백설', '해표', '샘표', '종가집', '비비고', '노브랜드', '피코크', '홈플러스', '이마트', '롯데', 
    '칠성', '코카', '펩시', '빙그레', '삼양', '팔도', '친환경', '유기농', 'GAP', '무농약', 
    '1+1', '2+1', '기획', '특가', '행사', '할인', '묶음', '증정', '초특가', '세일'
    # '국산', '수입'은 소고기 구분을 위해 여기서 제거하지 않고 로직에서 처리함
]

NAME_MAPPING = {
    '햇반': '즉석밥', '오뚜기밥': '즉석밥', '신라면': '라면', '진라면': '라면',
    '삼양라면': '라면', '너구리': '라면', '안성탕면': '라면', '짜파게티': '짜장라면',
    '비빔면': '라면', '불닭볶음면': '라면', '삼겹살': '돼지고기', '목살': '돼지고기',
    '앞다리': '돼지고기', '뒷다리': '돼지고기', '달걀': '계란', '특란': '계란',
    '왕란': '계란', '대란': '계란', '우유1L': '우유', '맛김': '조미김'
}

# 🚨 소고기(국산/수입) 구분 추가
FALLBACK_ESTIMATES = {
    '소고기(국산)': (100.0, 'g'), '소고기(수입)': (100.0, 'g'), '소고기': (100.0, 'g'),
    '돼지고기': (100.0, 'g'), '닭고기': (1000.0, 'g'),
    '두부': (300.0, 'g'), '브로콜리': (300.0, 'g'), '시금치': (300.0, 'g'),
    '대파': (500.0, 'g'), '상추': (200.0, 'g'), '깻잎': (30.0, 'g'),
    '콩나물': (300.0, 'g'), '무': (1500.0, 'g'), '배추': (2000.0, 'g'),
    '양배추': (2500.0, 'g'), '양파': (1500.0, 'g'), '감자': (100.0, 'g'),
    '고구마': (100.0, 'g'), '당근': (200.0, 'g'), '오이': (200.0, 'g'),
    '애호박': (300.0, 'g'), '버섯': (200.0, 'g'), '김치': (1000.0, 'g'),
    '쌀': (10000.0, 'g'), '우유': (900.0, 'ml'), '간장': (900.0, 'ml'),
    '식초': (900.0, 'ml'), '식용유': (900.0, 'ml'), '참기름': (320.0, 'ml'),
    '고추장': (1000.0, 'g'), '된장': (1000.0, 'g'), '설탕': (1000.0, 'g'),
    '소금': (1000.0, 'g'), '밀가루': (1000.0, 'g'), '부침가루': (1000.0, 'g'),
    '귤': (500.0, 'g'), '포도': (400.0, 'g'), '방울토마토': (500.0, 'g')
}

# ==========================================
# 2. 헬퍼 함수
# ==========================================

def normalize_name(name):
    if not name: return ""
    clean_name = name.strip()
    
    # 🚨 [신규] 소고기 원산지 분리 로직 (브랜드 제거 전에 수행)
    if '소고기' in clean_name or '등심' in clean_name or '양지' in clean_name:
        if any(k in clean_name for k in ['한우', '국산', '암소', '1++', '1등급']):
            return "소고기(국산)"
        if any(k in clean_name for k in ['수입', '호주', '미국', '와규', '척아이롤', '부채살']):
            return "소고기(수입)"
        # 원산지 불명확하면 그냥 소고기로 둠 (나중에 합쳐짐)

    # 1. 브랜드 제거
    for keyword in REMOVE_KEYWORDS:
        clean_name = clean_name.replace(keyword, '')
        
    clean_name = re.sub(r'\([^)]*\)', ' ', clean_name)
    clean_name = re.sub(r'\[[^]]*\]', ' ', clean_name)
    
    # 2. 숫자+단위 제거
    unit_patterns = r'(\d+[.,]?\d*)\s*(kg|g|ml|l|L|liter|개입|개|마리|단|봉|줄|포|팩|통|병|망|구|입|근|마|두|캔|박스|봉지|포장|손|송이|기)'
    clean_name = re.sub(unit_patterns, ' ', clean_name, flags=re.IGNORECASE)
    
    # 3. 단위 찌꺼기 제거 (배추 기 -> 배추)
    clean_name = re.sub(r'\s+[기통망단봉]$', '', clean_name)
    
    # 4. 특수문자 및 숫자 제거
    clean_name = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', clean_name)
    clean_name = re.sub(r'\s+\d+$', '', clean_name)
    clean_name = re.sub(r'^\d+\s+', '', clean_name)
    clean_name = re.sub(r'\s+', ' ', clean_name).strip()
    
    if clean_name in NAME_MAPPING:
        return NAME_MAPPING[clean_name]
    for key, val in NAME_MAPPING.items():
        if key in clean_name:
            return val

    return clean_name

def parse_unit_data(raw_name_str, normalized_name):
    s = raw_name_str.lower().replace(' ', '')
    s = re.sub(r'\([^)]*\)', '', s) 
    
    pattern = r'(\d+[.,]?\d*)\s*(kg|g|ml|l|liter|개입|개|마리|단|봉|줄|포|팩|통|병|망|구|입|근|마|두|캔|박스|봉지|포장|손)'
    match = re.search(pattern, s)
    
    amount = 1.0
    unit = 'piece'
    
    if match:
        raw_amount = float(match.group(1).replace(',', ''))
        raw_unit_text = match.group(2)
        
        if raw_unit_text in ['kg', 'kilo']: amount, unit = raw_amount * 1000.0, 'g'
        elif raw_unit_text in ['g', 'gram']: amount, unit = raw_amount, 'g'
        elif raw_unit_text in ['l', 'liter']: amount, unit = raw_amount * 1000.0, 'ml' 
        elif raw_unit_text in ['ml', 'cc']: amount, unit = raw_amount, 'ml'
        elif raw_unit_text == '손': amount, unit = raw_amount * 2.0, 'piece'
        else: amount, unit = raw_amount, 'piece'
    
    # Fallback 적용
    for key, (est_amount, est_unit) in FALLBACK_ESTIMATES.items():
        if key in normalized_name:
            if unit == 'piece': return amount * est_amount, est_unit
            return amount, unit
            
    return amount, unit

# ==========================================
# 3. 메인 로직
# ==========================================

def fetch_and_process_data(cursor, conn):
    print(f"--- '{TARGET_MARKET_KEYWORD}' 데이터 처리 시작 ---")
    
    url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/ListNecessariesPricesService/1/1000/"
    try:
        resp = requests.get(url, timeout=20)
        data = resp.json().get('ListNecessariesPricesService', {}).get('row', [])
    except Exception as e:
        print(f"API 에러: {e}")
        return

    # 그룹핑 데이터 (이름별로 가격 리스트 모으기)
    grouped_data = defaultdict(lambda: {'p_piece': [], 'p_100g': [], 'p_100ml': [], 'unit': 'piece', 'market': '', 'gu': ''})

    for item in data:
        raw_name = item.get("A_NAME") 
        raw_price = float(item.get("A_PRICE", 0))
        market_name = item.get("M_NAME", "")
        gu_name = item.get("M_GU_NAME", "")

        if TARGET_MARKET_KEYWORD not in market_name: continue
        if not raw_name or raw_price <= 0: continue 
        if any(bad in raw_name for bad in NON_FOOD_ITEMS): continue 

        # 1. 정규화 (여기서 소고기(국산)/(수입)으로 갈라짐)
        norm_name = normalize_name(raw_name)
        if not norm_name or norm_name.isdigit(): continue

        # 2. 단위 파싱
        amount, unit = parse_unit_data(raw_name, norm_name)
        
        # 3. 데이터 모으기 (평균을 위해)
        entry = grouped_data[norm_name]
        entry['unit'] = unit
        entry['market'] = market_name
        entry['gu'] = gu_name
        
        if unit == 'g':
            entry['p_100g'].append((raw_price / amount) * 100.0)
        elif unit == 'ml':
            entry['p_100ml'].append((raw_price / amount) * 100.0)
        else:
            entry['p_piece'].append(raw_price / amount)

    saved_count = 0
    today = date.today()

    # 4. 평균 내서 저장
    for name, data in grouped_data.items():
        avg_piece = sum(data['p_piece']) / len(data['p_piece']) if data['p_piece'] else None
        avg_100g = sum(data['p_100g']) / len(data['p_100g']) if data['p_100g'] else None
        avg_100ml = sum(data['p_100ml']) / len(data['p_100ml']) if data['p_100ml'] else None
        
        final_unit = data['unit']

        # Ingredients 테이블 확인/업데이트
        cursor.execute("SELECT id, default_unit FROM ingredients WHERE name = %s", (name,))
        row = cursor.fetchone()
        
        if row:
            ingredient_id = row['id']
            # 기존 단위보다 g/ml가 우선
            if row['default_unit'] != final_unit and final_unit in ['g', 'ml']:
                cursor.execute("UPDATE ingredients SET default_unit = %s WHERE id = %s", (final_unit, ingredient_id))
        else:
            try:
                cursor.execute("INSERT INTO ingredients (name, default_unit) VALUES (%s, %s)", (name, final_unit))
                ingredient_id = cursor.lastrowid
            except mysql.connector.Error:
                cursor.execute("SELECT id FROM ingredients WHERE name = %s", (name,))
                ingredient_id = cursor.fetchone()['id']

        # Price 테이블 저장
        sql_price = """
            INSERT INTO ingredient_prices 
            (ingredient_id, currency, price_per_piece, price_per_100g, price_per_100ml, source, region, effective_date)
            VALUES (%s, 'KRW', %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                price_per_piece = VALUES(price_per_piece),
                price_per_100g = VALUES(price_per_100g),
                price_per_100ml = VALUES(price_per_100ml),
                source = VALUES(source),
                region = VALUES(region);
        """
        try:
            cursor.execute(sql_price, (
                ingredient_id, avg_piece, avg_100g, avg_100ml, 
                data['market'], data['gu'], today
            ))
            saved_count += 1
        except mysql.connector.Error:
            pass

    conn.commit()
    print(f"✅ 총 {saved_count}개 품목 (중복 통합/원산지 분리) 저장 완료.")

def calculate_average_prices(cursor, conn):
    print("--- 재료별 평균 가격 업데이트 ---")
    sql = """
        UPDATE ingredients i
        JOIN (
            SELECT ingredient_id, 
                   AVG(COALESCE(price_per_100g, price_per_100ml, price_per_piece)) as avg_p
            FROM ingredient_prices
            WHERE effective_date = CURDATE()
            GROUP BY ingredient_id
        ) p ON i.id = p.ingredient_id
        SET i.avg_price = p.avg_p
    """
    cursor.execute(sql)
    conn.commit()
    print("✅ 완료.")

def main():
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cur = conn.cursor(dictionary=True)
        
        # 오늘 데이터 리셋
        cur.execute("DELETE FROM ingredient_prices WHERE effective_date = CURDATE()")
        conn.commit()

        fetch_and_process_data(cur, conn)
        calculate_average_prices(cur, conn)
        
        print("\n🎉 최종 완료!")

    except Exception as e:
        print(f"Error: {e}")
    finally:
        if conn: conn.close()

if __name__ == "__main__":
    main()

--- '이마트' 데이터 처리 시작 ---
✅ 총 66개 품목 (중복 통합/원산지 분리) 저장 완료.
--- 재료별 평균 가격 업데이트 ---
✅ 완료.

🎉 최종 완료!


In [4]:
import requests

API_KEY = "e503e06ce9147b59d04cb64c2eee1045c66ac1993f32c31806732a85a33db5c4"

url = "http://apis.data.go.kr/1130000/PriceInfoService/getListPriceInfo"
params = {
    "serviceKey": API_KEY,
    "pageNo": 1,
    "numOfRows": 5,
    "resultType": "json"
}

resp = requests.get(url, params=params)
print(resp.json())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [6]:
import requests

API_KEY = "e503e06ce9147b59d04cb64c2eee1045c66ac1993f32c31806732a85a33db5c4"

url = "https://openapi.price.go.kr/openApiImpl/ProductPriceInfoService/getProductInfoList"
params = {
    "serviceKey": API_KEY,
    "pageNo": 1,
    "numOfRows": 3,
}

resp = requests.get(url, params=params)
print("상태코드:", resp.status_code)
print("응답내용:", resp.text[:1000])

ConnectTimeout: HTTPSConnectionPool(host='openapi.price.go.kr', port=443): Max retries exceeded with url: /openApiImpl/ProductPriceInfoService/getProductInfoList?serviceKey=e503e06ce9147b59d04cb64c2eee1045c66ac1993f32c31806732a85a33db5c4&pageNo=1&numOfRows=3 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000221BB177250>, 'Connection to openapi.price.go.kr timed out. (connect timeout=None)'))